# Convert outputs to yearly zarr files

In [1]:
import os
import sys
import zarr
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr

In [2]:
config_name = os.path.realpath('verif_config.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [3]:
# model_name = 'swin-wrf'
# source_dir = conf[model_name]['save_loc_gather']

In [3]:
filenames = sorted(glob('/glade/derecho/scratch/ksha/DWC/RAW_OUTPUT/CONUS_GP_diag_L/*/*.nc'))

In [9]:
ds_collection = []

for fn in filenames[:3]:
    ds = xr.open_dataset(fn)
    ds_collection.append(ds)

In [10]:
ds_merge = xr.concat(ds_collection, 'time')

In [12]:
# zarr encodings
dict_encoding = {}
varnames = list(ds_merge.keys())
# varname_3D = ['WRF_precip', 'WRF_PWAT', 'WRF_radar_composite', 'WRF_TCC', 'WRF_OLR']

chunk_size_3d = dict(chunks=(1, 336, 336))
chunk_size_4d = dict(chunks=(1, 16, 336, 336))
compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

for i_var, var in enumerate(varnames):
    # if var in varname_4D:
    #     dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
    # else:
    dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

In [13]:
save_name = '/glade/campaign/ral/hap/ksha/DWC/GATHER/CONUS_GP_ERA5_diag/precip_2020-01-01T00Z.zarr'
# ds_merge.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)